# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by the Croissant schema at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note**: all entities are referenced by their `@id` fields.

In [ ]:
# List all record sets and their fields by @id for precise referencing

print("Available record sets and their fields:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"\nRecord set name: {rs.name}")
    print(f"  @id: {rs.id}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for f in rs.fields:
            print(f"    - {f.name} (@id: {f.id}, type: {getattr(f, 'data_type', 'unknown')})")

## 3. Data Extraction
Load each record set into a DataFrame for analysis using their `@id`s.

In [ ]:
# We'll extract all available record sets by @id

dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

print("Extracted dataframes (by record set @id):")
for rsid in dataframes:
    print(f"@id: {rsid}, columns: {list(dataframes[rsid].columns)}")

# Select one record set for exploration:
if dataframes:
    selected_rs_id = list(dataframes.keys())[0]
    print(f"\nSample preview of record set '{selected_rs_id}':")
    display(dataframes[selected_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process and explore the first record set using one of its numeric fields and a grouping variable. All references are via the entities' `@id`s.

_If in your use-case the record set lacks numeric fields, adapt by choosing an appropriate column._

In [ ]:
# Identify numeric fields in the selected record set
selected_fields = None
for rs in dataset.record_sets:
    if rs.id == selected_rs_id:
        selected_fields = rs.fields
        break

numeric_field_id = None
group_field_id = None
if selected_fields:
    for f in selected_fields:
        dtype = getattr(f, 'data_type', '').lower()
        if dtype in ['integer','float','number'] and numeric_field_id is None:
            numeric_field_id = f.id
        # Heuristically pick a grouping field, e.g. 'sex', 'anatomy', etc.:
        if (('sex' in f.name.lower()) or ('site' in f.name.lower()) or ('anatomical' in f.name.lower())) and group_field_id is None:
            group_field_id = f.id

df = dataframes[selected_rs_id]

if numeric_field_id and numeric_field_id in df.columns:
    # For numeric processing, set a threshold
    threshold = 10
    mask = pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold
    filtered_df = df[mask].copy()
    print(f"Filtered records ({numeric_field_id} > {threshold}):")
    display(filtered_df[[numeric_field_id]].head())
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') -
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a group_field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df)
else:
    print("No numeric field found for EDA in this record set.")

## 5. Visualization
Visualize the distribution of the selected numeric field and explore group-wise differences, if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to load and analyze the FAIR^2 Clinical Colorectal Cancer dataset by referencing all entities by their `@id`, as recommended for reproducibility and schema-driven workflows.

Key findings and notebook steps:
- Dataset metadata and Croissant schema successfully loaded.
- Dataframe extraction for each record set by unique `@id`.
- Sample EDA and visualization for a selected numeric field and grouping variable.

You can further extend this analysis by:
- Exploring all record sets and joining related data by IDs.
- Performing statistical analyses or ML model training with the curated DataFrames.
- Building FAIR pipelines powered by Croissant records and schemas.